# Week 5 Exercise — Chiku: Stayez Property Knowledge Worker

**Student:** Vagz1216 (Martin Kamau) | **Team:** Euclid | **Week 5**

---

## Project Overview

This notebook implements a complete **RAG (Retrieval Augmented Generation)** pipeline applied to the real Stayez platform (stayez.co.ke). Instead of a generic personal knowledge base, I built a **Stayez Property Knowledge Worker** — an AI booking assistant called **Chiku** that can answer guest queries by semantically searching a curated knowledge base of actual Stayez properties, experiences, and services.

### The Business Problem
Stayez currently has hardcoded property data in a Python dictionary. As the platform scales to 500+ listings, Chiku cannot find the right property for complex guest queries like:
- *"I need a romantic apartment in Nairobi under KSh 7,000 with a pool"*
- *"Something good for a family of 5 with parking"*
- *"What outdoor experiences do you have near a volcano?"*

**RAG solves this.** Each property is a markdown file. Any new listing gets added as a new `.md` file. Chiku finds the right matches through semantic vector search.

### Week 5 Concepts Demonstrated
| Concept | Implementation |
|---|---|
| Document Loading | LangChain `DirectoryLoader` per folder type |
| Text Chunking | `RecursiveCharacterTextSplitter` (1000 chars, 200 overlap) |
| Embeddings | `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` — free & local |
| Vector Store | `Chroma` persistent database |
| Visualization | t-SNE 2D + 3D Plotly scatter plots |
| RAG Chat | Retriever + Groq/Gemini LLM injection |
| UI | Gradio `ChatInterface` for Chiku |


In [ ]:
# Cell 1: Imports

import os
import glob
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from sklearn.manifold import TSNE

# LangChain
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Imports loaded successfully")

In [ ]:
# Cell 2: Configuration & API Keys

load_dotenv(find_dotenv(), override=True)

GROQ_API_KEY   = os.getenv('GROQ_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

for name, key in [("Groq", GROQ_API_KEY), ("Gemini", GEMINI_API_KEY)]:
    print(f"{name}: {'Found' if key else 'NOT SET'}")

# Groq is our primary LLM (free and fast)
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Gemini as fallback
gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Settings
DB_NAME           = "stayez_vector_db"
KNOWLEDGE_BASE    = Path("stayez-knowledge-base")
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"   # Free HuggingFace model, runs locally
CHUNK_SIZE        = 1000
CHUNK_OVERLAP     = 200

print(f"\nKnowledge base path: {KNOWLEDGE_BASE.resolve()}")
print(f"Vector DB: {DB_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## Part A: Load & Chunk the Stayez Knowledge Base

The knowledge base is organized into 3 folders:
- `properties/` — Individual property listing files (7 properties)
- `experiences/` — Activity and experience listings  
- `services/` — Concierge and support services

Each folder becomes a document `type` tag in the vector store metadata.


In [ ]:
# Cell 3: Load documents from the knowledge base folders

documents = []

for folder in KNOWLEDGE_BASE.iterdir():
    if not folder.is_dir():
        continue
    doc_type = folder.name
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = Path(doc.metadata["source"]).name
    documents.extend(folder_docs)
    print(f"  - Loaded {len(folder_docs)} docs from '{doc_type}/'")

print(f"\nTotal documents loaded: {len(documents)}")

# Preview
print(f"\nFirst document (excerpt):")
print(documents[0].page_content[:300])
print(f"Metadata: {documents[0].metadata}")

In [ ]:
# Cell 4: Split into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

print(f"Documents split into {len(chunks)} chunks")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

## Part B: Embed Chunks & Build Chroma Vector Store

We use **HuggingFace's `all-MiniLM-L6-v2`** model to convert each chunk into a 384-dimensional vector. This model:
- Runs **100% locally** on CPU (no API calls)
- Is **completely free** with no rate limits
- Downloads automatically on first use (~90 MB)
- Provides excellent semantic search quality for English text


In [ ]:
# Cell 5: Create embeddings and build Chroma vector store

print("Loading HuggingFace embedding model (downloads ~90MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Embedding model ready!")

# Delete existing collection to start fresh
if Path(DB_NAME).exists():
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Cleared existing vector store'{DB_NAME}'")

# Build and persist the vector store
print("\nBuilding Chroma vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

collection = vectorstore._collection
count = collection.count()
sample_emb = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

print(f"\nVector store created:")
print(f"  - Total vectors stored: {count:,}")
print(f"  - Embedding dimensions: {len(sample_emb):,} (all-MiniLM-L6-v2)")
print(f"  - Persisted to: ./{DB_NAME}/")

## Part C: Visualize the Vector Space

We use **t-SNE** (t-Distributed Stochastic Neighbor Embedding) to compress the 384-dimensional vectors down to 2D and 3D for visualization. Each colored dot is a chunk, color-coded by its document category (property, experience, or service).

> **What to look for:** If RAG is working correctly, similar content clusters together. Property chunks should group together, experience chunks together, etc. But within the properties cluster, romantic/luxury properties should be near each other and budget properties near each other.


In [ ]:
# Cell 6: Prepare data for visualization

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors   = np.array(result["embeddings"])
docs_text = result["documents"]
metadatas = result["metadatas"]
doc_types = [m["doc_type"] for m in metadatas]

# Color map by document type
COLOR_MAP = {"properties": "royalblue", "experiences": "mediumseagreen", "services": "tomato"}
colors = [COLOR_MAP.get(t, "gray") for t in doc_types]

print(f"Data ready for visualization: {len(vectors)} vectors")
from collections import Counter
print(f"Breakdown: {dict(Counter(doc_types))}")

In [ ]:
# Cell 7: 2D t-SNE Visualization

tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_2d = tsne_2d.fit_transform(vectors)

fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0],
    y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=10, color=colors, opacity=0.85, line=dict(width=1, color="white")),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_2d.update_layout(
    title="2D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=850, height=600,
    template="plotly_white"
)
fig_2d.show()

In [ ]:
# Cell 8: 3D t-SNE Visualization

tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_3d = tsne_3d.fit_transform(vectors)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0],
    y=reduced_3d[:, 1],
    z=reduced_3d[:, 2],
    mode="markers",
    marker=dict(size=6, color=colors, opacity=0.85),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_3d.update_layout(
    title="3D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900, height=700
)
fig_3d.show()

## Part D: Build Chiku — The Stayez RAG Booking Assistant

Now we wire the retriever and LLM together to create **Chiku**.

The flow for every guest message:
1. Chiku receives the guest's question (e.g. *"I need a romantic apartment"*)
2. The **Retriever** converts the question into a vector and finds the 5 most semantically similar chunks from the Chroma database
3. The retrieved chunks are injected into Chiku's **system prompt** as context
4. **Groq (Llama 3.3 70B)** generates the final personalized answer using both the context and the guest's question


In [ ]:
# Cell 9: Chiku RAG System Setup

# k=3 keeps context concise (avoids token limit on Groq free tier)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

CHIKU_SYSTEM = """\
You are Chiku, a warm and knowledgeable guest assistant for Stayez (stayez.co.ke), \
a curated short-stay booking platform based in Kenya.

Your job is to help guests find the perfect property, experience, or service.
You are enthusiastic, friendly, and always recommend booking via stayez.co.ke.
All prices are in Kenyan Shillings (KSh). Always present prices clearly.

Use the context below to answer the guest's question accurately and concisely.
If information is not in the context, say so and suggest the guest visits stayez.co.ke.

RELEVANT STAYEZ CONTEXT:
{context}
"""

def build_messages(message: str, history: list, system_prompt: str) -> list:
    """Build a clean OpenAI-compatible message list.

    Handles any Gradio history format (tuples OR message dicts).
    IMPORTANT: strips any extra fields (e.g. Gradio 5 adds 'metadata')
    since Groq and Gemini only accept 'role' + 'content'.
    Caps history to last 4 messages (2 full turns) to stay within token limits.
    """
    normalised = []
    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            # Whitelist only role + content — drop 'metadata' and any other Gradio-added fields
            normalised.append({"role": item["role"], "content": item["content"]})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            if u: normalised.append({"role": "user",      "content": str(u)})
            if b: normalised.append({"role": "assistant", "content": str(b)})
    normalised = normalised[-4:]  # keep last 2 turns only
    return [{"role": "system", "content": system_prompt}] + normalised + [{"role": "user", "content": message}]

def chiku_chat(message: str, history: list) -> str:
    # Step 1: Semantic retrieval from Chroma
    docs = retriever.invoke(message)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Step 2: Build messages (clean, no extra fields)
    system_prompt = CHIKU_SYSTEM.format(context=context)
    messages = build_messages(message, history, system_prompt)

    # Step 3: Try Groq — llama-3.1-8b-instant (20,000 TPM free tier)
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as groq_err:
        print(f"[Groq error] {type(groq_err).__name__}: {groq_err}")

    # Step 4: Fallback to Gemini (gemini-2.0-flash-lite)
    try:
        response = gemini_client.chat.completions.create(
            model="gemini-2.0-flash-lite",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as gemini_err:
        print(f"[Gemini error] {type(gemini_err).__name__}: {gemini_err}")
        return "Chiku is temporarily unavailable. Please visit stayez.co.ke directly."

print("Chiku is ready!")

# Self-test: 2 turns (ensures both Groq and history handling work before launching UI)
r1 = chiku_chat("What is your most popular property?", [])
print(f"Turn 1 OK: {r1[:200]}")

hist_test = [{"role": "user", "content": "What is your most popular property?"}, {"role": "assistant", "content": r1}]
r2 = chiku_chat("I need something romantic under KSh 7,000", hist_test)
print(f"Turn 2 OK: {r2[:200]}")

In [ ]:
# Cell 10: Launch Chiku on Gradio

with gr.Blocks(theme=gr.themes.Soft(), title="Chiku — Stayez AI Booking Assistant") as demo:
    gr.Markdown(
        """## Chiku — Your Stayez Booking Assistant
Powered by RAG + Groq. Ask me about properties, experiences, and services at **stayez.co.ke**.

*Try: "I need a romantic apartment under KSh 7,000" or "What outdoor experiences do you have?"*
"""
    )
    chat = gr.ChatInterface(
        fn=chiku_chat,
        type="messages",
        examples=[
            "What is your most popular property?",
            "I need a romantic place in Nairobi for 2 nights, budget KSh 6,000",
            "Do you have properties for a family of 5?",
            "What outdoor experiences do you offer?",
            "Can you arrange an airport pickup?",
        ],
        chatbot=gr.Chatbot(type="messages", height=450, avatar_images=[None, "https://stayez.co.ke/wp-content/uploads/2023/08/stayez-logo.jpg"])
    )

demo.launch(inbrowser=True)

---
## Summary

### What We Built
A complete, production-ready RAG pipeline for the Stayez platform:

1. **Stayez Knowledge Base** — 10 markdown files (7 properties, 2 experiences, 1 service) as the dynamic, scaleable data source.
2. **LangChain Document Loading** — `DirectoryLoader` automatically tagged each document with its folder type (property/experience/service).
3. **`RecursiveCharacterTextSplitter`** — Chunked documents intelligently with 200-character overlap to avoid losing context at boundaries.
4. **Free Embeddings** — `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` runs locally at zero cost, converting text to 384-dimensional vectors.
5. **Chroma Vector Store** — Persisted the vector database to disk so it only needs to be built once.
6. **t-SNE Visualization** — Proved that the vector space organizes correctly: property chunks cluster together, experience chunks cluster together.
7. **Chiku RAG Chat** — 5-step pipeline: guest message → vector search → context injection → Groq LLM → friendly answer.
8. **Gradio UI** — Clean chat interface with example prompts for quick testing.

### What This Enables for Stayez
When the Stayez team adds a new property, they simply drop a new `.md` file into `stayez-knowledge-base/properties/` and rebuild the vector store. The RAG system handles all matching automatically — no hardcoded logic required.
